# Lab 2 — Comparing LLM Responses

This lab builds on my first API experiments by comparing how different OpenAI models respond to the same question.

**Focus:** model comparison, reusable functions, response collection, and LLM-based evaluation.

*Based on concepts from Ed Donner's Agentic AI course, with my own modifications.*


## 1. Setup

I’m using Python environment variables for API access and the OpenAI client for the model calls.


In [1]:
# os + dotenv: read the API key from the environment, not from this notebook.
# OpenAI: make chat completions. json: parse the judge model's ranking later.
# Markdown/display: show model replies in a readable way in Jupyter.

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display


In [2]:
# Load .env into the process. override=True so this notebook's key wins if one was already set.
load_dotenv(override=True)


True

In [3]:
# This is just to chcek for other keys in my environment. But since i am using only OpenAI. iT WAS JUST A TEST.
# Confirm keys loaded without printing the secret. This notebook only calls OpenAI.
# the other checks are leftover from the course lab and are optional here.

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print("OpenAI API Key is available!!!!! Yeeee")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print("Anthropic API Key is available.")
else:
    print("Anthropic API Key not set (So sad. Need to work harder)")

if google_api_key:
    print("Google API Key is available.")
else:
    print("Google API Key not set (so sad. Need to work smarter)")

if deepseek_api_key:
    print("DeepSeek API Key is available.")
else:
    print("DeepSeek API Key not set (so sad.)")

if groq_api_key:
    print("Groq API Key is available.")
else:
    print("Groq API Key not set (so sad.)")

if grok_api_key:
    print("Grok API Key is available.")
else:
    print("Grok API Key not set (Optional But so sad.)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}") 
else:
    print("OpenRouter API Key not set (Optional. So sad. I think I actually nneedd more money)")


OpenAI API Key is available!!!!! Yeeee
Anthropic API Key not set (So sad. Need to work harder)
Google API Key not set (so sad. Need to work smarter)
DeepSeek API Key not set (so sad.)
Groq API Key not set (so sad.)
Grok API Key not set (Optional But so sad.)
OpenRouter API Key not set (Optional. So sad. I think I actually nneedd more money)


## 2. Generate a question

Instead of comparing models on different prompts, I use one challenging question so the responses can be compared on the same task.


In [4]:
# Ask a model to invent ONE short, thought-provoking question.
# I generate the test question instead of writing it myself, then reuse it for every comparison.

request = """
Please come up with a challenging, nuanced question with a succinct answer
that I can ask several versions of ChatGPT, so I can compare how they think.
Not a mathematical puzzle, but a thought-provoking question that needs intelligent insight.
Include in your question that the answer must be short.
"""


request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]


In [5]:
# Sanity-check: chat APIs take a list of {role, content} dicts, not a bare string.
messages


[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer\nthat I can ask several versions of ChatGPT, so I can compare how they think.\nNot a mathematical puzzle, but a thought-provoking question that needs intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [6]:
# Create the OpenAI client (reads OPENAI_API_KEY).
# gpt-5.6-sol writes the shared test question; later cells only change the model ID.

openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5.6-sol", messages=messages)
question = response.choices[0].message.content

display(Markdown(question))


When, if ever, is it ethically justified to hide a true fact from the public—not because disclosure would cause immediate harm, but because people are likely to misunderstand and misuse it? Answer in no more than 75 words.

## 3. Compare different OpenAI models

I kept the provider constant and changed only the model ID. This makes the comparison focused on how different model versions approach the same question.


In [7]:
# ChatGPT versions I will compare, cheapest to stronger.
# API IDs are lowercase with hyphens (not display names like "GPT-5.4 Mini").

model_name = "gpt-5.4-nano"
model_name0 = "gpt-5.4-mini"
model_name1 = "gpt-5.5"
model_name2 = "gpt-5.6-luna"
model_name3 = "gpt-5.6-terra"


In [8]:
# competitors/answers: parallel lists so I can zip model ID with its reply for judging.
# messages now holds the generated question, the same prompt for every model.

competitors = []
answers = []
messages = [{"role": "user", "content": question}]


In [9]:
# One helper so every comparison cell stores results the same way (needed by the judge later).

def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))


In [10]:
# Smallest/cheapest model in the set. Same messages; only the model ID changes.
# reasoning_effort="none" keeps this comparison closer to a direct answer.

model_name = "gpt-5.4-nano"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)



Ethically, hiding a true fact is justified only in exceptional circumstances: when credible evidence shows likely misunderstanding will foreseeably cause serious, disproportionate harm, and no less deceptive alternative (context, caveats, education, gradual release) would sufficiently reduce the risk. It also requires transparency about why withholding is temporary, evidence-based judgment, minimal secrecy, and accountability. In general, truth should be shared with proper framing rather than concealed.

In [11]:
# Next size up: still cheap, usually a bit stronger than nano. Same question.

model_name = "gpt-5.4-mini"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)


Only exceptionally. Withholding a true fact can be ethical when disclosure would create a high, foreseeable risk of serious harm, the public has no realistic way to interpret it safely, and the information is not essential for accountability or informed consent. Even then, secrecy should be temporary, narrow, and paired with a plan for safer disclosure. As a default, truth should be published; fear of misuse alone is usually not enough.

In [12]:
# Mid-tier GPT-5.5, still OpenAI-only so provider is not a confounder.

model_name = "gpt-5.5"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)


Ethically, hiding a true fact is justified only rarely: when misunderstanding is highly likely, the resulting misuse would cause serious, foreseeable harm, and no less deceptive alternative—context, education, phased release, expert framing, or safeguards—would suffice. Mere fear that “the public can’t handle it” is paternalistic and dangerous. The burden of proof lies on those withholding it, with transparency about the withholding process, independent oversight, and a plan for eventual disclosure.

In [13]:
# GPT-5.6 Luna: cost-focused 5.6 variant, still with reasoning_effort none.

model_name = "gpt-5.6-luna"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)


It may be ethically justified only under exceptional conditions: the fact is genuinely dangerous if broadly misused, misunderstanding is highly probable, harms are serious and foreseeable, and no safer alternative—context, education, delayed release, or restricted access—would work. Secrecy should be temporary, narrowly tailored, independently reviewed, and accountable. Mere offensiveness, embarrassment, political inconvenience, or paternalistic distrust is insufficient; competent adults generally deserve truth and the opportunity to reason about it.

In [14]:
# GPT-5.6 Terra: stronger 5.6 tier. low reasoning to see if extra thinking changes the answer.

model_name = "gpt-5.6-terra"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="low")
answer = response.choices[0].message.content

record(model_name, answer)


Only rarely: when misuse is highly foreseeable, serious, and not preventable through context, education, or limited disclosure; when withholding is temporary and proportionate; and when accountable, independent oversight exists. “People may misunderstand” alone is insufficient—publics deserve respect and opportunities to learn. The burden lies with those withholding information to show that secrecy prevents greater harm without chiefly protecting power, reputation, or convenience.

## 4. Organize the responses

Once the responses are collected, I organize them so they can be reviewed and evaluated as a group.


In [15]:
# Confirm I collected one reply per model before judging (expect 5).

print(len(competitors))
print(competitors)
print(answers)


5
['gpt-5.4-nano', 'gpt-5.4-mini', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-terra']
['Ethically, hiding a true fact is justified only in exceptional circumstances: when credible evidence shows likely misunderstanding will foreseeably cause serious, disproportionate harm, and no less deceptive alternative (context, caveats, education, gradual release) would sufficiently reduce the risk. It also requires transparency about why withholding is temporary, evidence-based judgment, minimal secrecy, and accountability. In general, truth should be shared with proper framing rather than concealed.', 'Only exceptionally. Withholding a true fact can be ethical when disclosure would create a high, foreseeable risk of serious harm, the public has no realistic way to interpret it safely, and the information is not essential for accountability or informed consent. Even then, secrecy should be temporary, narrow, and paired with a plan for safer disclosure. As a default, truth should be published; fear of mi

In [16]:
# zip keeps each model ID next to its own answer (lists stay in call order).

for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5.4-nano

Ethically, hiding a true fact is justified only in exceptional circumstances: when credible evidence shows likely misunderstanding will foreseeably cause serious, disproportionate harm, and no less deceptive alternative (context, caveats, education, gradual release) would sufficiently reduce the risk. It also requires transparency about why withholding is temporary, evidence-based judgment, minimal secrecy, and accountability. In general, truth should be shared with proper framing rather than concealed.
Competitor: gpt-5.4-mini

Only exceptionally. Withholding a true fact can be ethical when disclosure would create a high, foreseeable risk of serious harm, the public has no realistic way to interpret it safely, and the information is not essential for accountability or informed consent. Even then, secrecy should be temporary, narrow, and paired with a plan for safer disclosure. As a default, truth should be published; fear of misuse alone is usually not enough

In [17]:
# Combine all competitors' responses into one text block, labeling each response
# so the AI judge can easily compare and rank them.
# Basically. This code takes the responses stored in answers and puts them together into one piece of text.

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"


In [18]:
# Display the combined responses before evaluation.

print(together)


# Response from competitor 1

Ethically, hiding a true fact is justified only in exceptional circumstances: when credible evidence shows likely misunderstanding will foreseeably cause serious, disproportionate harm, and no less deceptive alternative (context, caveats, education, gradual release) would sufficiently reduce the risk. It also requires transparency about why withholding is temporary, evidence-based judgment, minimal secrecy, and accountability. In general, truth should be shared with proper framing rather than concealed.

# Response from competitor 2

Only exceptionally. Withholding a true fact can be ethical when disclosure would create a high, foreseeable risk of serious harm, the public has no realistic way to interpret it safely, and the information is not essential for accountability or informed consent. Even then, secrecy should be temporary, narrow, and paired with a plan for safer disclosure. As a default, truth should be published; fear of misuse alone is usually n

## 5. Use an LLM as a judge

The next step is to evaluate the responses rather than relying only on manual inspection.

I create a judging prompt that asks another model to rank the responses against the same criteria.


In [19]:
# Build a structured prompt for an AI judge to evaluate and rank multiple model version responses. (GPT-6)
# The judge compares each response for clarity and strength of argument, then returns
# the competitors ranked from best to worst in a JSON format. 
#That is what the code basically does

judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [20]:
# Inspect the judging prompt before sending it to the evaluator.

print(judge)


You are judging a competition between 5 competitors.
Each model has been given this question:

When, if ever, is it ethically justified to hide a true fact from the public—not because disclosure would cause immediate harm, but because people are likely to misunderstand and misuse it? Answer in no more than 75 words.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

Ethically, hiding a true fact is justified only in exceptional circumstances: when credible evidence shows likely misunderstanding will foreseeably cause serious, disproportionate harm, and no less deceptive alternative (context, caveats, education, gradual release) would sufficiently reduce the risk. It also requi

In [21]:
# Prepare the judging prompt as an API message.

judge_messages = [{"role": "user", "content": judge}]


## 6. Ranking the results

I use another model to evaluate the responses and return a structured ranking.


# Use a separate model to evaluate and rank the responses.
## And now for 6-astra!

Branded as "The most truth-seeking large language model in the world".. NOT REALLY BUT GIVEN ITS THE LATEST VERSION WE CAN SAY THAT

In [22]:
# Parse the evaluator's JSON result and display the final ranking.

# Judgement time!
# We are using gpt-6-astra is "The most truth-seeking large language model in the world."
# # NOT REALLY BUT GIVEN ITS THE LATEST VERSION WE CAN SAY THAT



model_name = "gpt-6-astra"

response = openai.chat.completions.create(model=model_name, messages=judge_messages)
results = response.choices[0].message.content
print(results)


{"results":["3","5","2","4","1"]}


In [23]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")


# We can see "gpt-5.6-terra" is mostly on the high and thats probably due to the reasoning we made low and not none.

Rank 1: gpt-5.5
Rank 2: gpt-5.6-terra
Rank 3: gpt-5.4-mini
Rank 4: gpt-5.6-luna
Rank 5: gpt-5.4-nano


## Key Takeaway

This lab helped me move from making individual LLM calls to designing a small **multi-model evaluation workflow**.

The pattern is:

**Generate → Compare → Evaluate → Rank**

This is an early example of how multiple LLM calls can work together as part of an agentic system.
